# 📊 Exploratory Data Analysis - AI Expense & Sales Analyzer

This notebook performs end-to-end **Exploratory Data Analysis (EDA)** on multi-year sales transactions, operating expenses, profit margins, product categories, and geographical regions.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3

# Graphics styling
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
print("Libraries loaded successfully.")

## 1. Data Ingestion
Load cleaned dataset from processed CSV.

In [ ]:
data_path = '../data/processed/cleaned_sales_data.csv'
if not os.path.exists(data_path):
    data_path = 'data/processed/cleaned_sales_data.csv'

df = pd.read_csv(data_path)
df['Date'] = pd.to_datetime(df['Date'])
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

## 2. Dataset Health & Summary Statistics

In [ ]:
print("--- Missing Values Audit ---")
print(df.isnull().sum())

print("\n--- Descriptive Statistics ---")
df[['Revenue', 'COGS', 'Marketing_Expense', 'Total_Expense', 'Net_Profit', 'Profit_Margin_Pct']].describe().round(2)

## 3. Revenue & Expense Breakdown by Category

In [ ]:
cat_summary = df.groupby('Product_Category').agg({
    'Revenue': 'sum',
    'Total_Expense': 'sum',
    'Net_Profit': 'sum'
}).reset_index().sort_values(by='Revenue', ascending=False)

cat_summary['Net_Margin_%'] = (cat_summary['Net_Profit'] / cat_summary['Revenue'] * 100).round(2)
cat_summary

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(cat_summary))
width = 0.35

ax.bar(x - width/2, cat_summary['Revenue'] / 1e6, width, label='Revenue ($M)', color='#2E7D32')
ax.bar(x + width/2, cat_summary['Total_Expense'] / 1e6, width, label='Total Expense ($M)', color='#C62828')

ax.set_ylabel('USD ($ Millions)')
ax.set_title('Revenue vs Expenses by Category')
ax.set_xticks(x)
ax.set_xticklabels(cat_summary['Product_Category'])
ax.legend()
plt.tight_layout()
plt.show()

## 4. Monthly Financial Trends

In [ ]:
monthly = df.groupby('Year_Month').agg({
    'Revenue': 'sum',
    'Total_Expense': 'sum',
    'Net_Profit': 'sum'
}).reset_index()

plt.figure(figsize=(12, 5))
plt.plot(monthly['Year_Month'], monthly['Revenue'] / 1e3, label='Revenue ($K)', marker='o', color='#1565C0', linewidth=2)
plt.plot(monthly['Year_Month'], monthly['Total_Expense'] / 1e3, label='Expenses ($K)', marker='s', color='#D32F2F', linewidth=2)
plt.plot(monthly['Year_Month'], monthly['Net_Profit'] / 1e3, label='Net Profit ($K)', marker='^', color='#2E7D32', linewidth=2)
plt.xticks(rotation=45)
plt.title('Monthly Financial Performance Trajectory')
plt.ylabel('Amount in Thousands ($K)')
plt.legend()
plt.tight_layout()
plt.show()

## 5. Correlation Analysis

In [ ]:
num_cols = ['Revenue', 'COGS', 'Marketing_Expense', 'Shipping_Expense', 'Operating_Expense', 'Total_Expense', 'Net_Profit']
corr = df[num_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='Blues', fmt='.2f', linewidths=0.5)
plt.title('Financial Metrics Correlation Heatmap')
plt.tight_layout()
plt.show()